# Project Code Reference: the tested foundation the weekly notebooks build on

The weekly notebooks import this logic from `src/equibirth/` so the test suite guards it against drift, and inline the rest in per-notebook SETUP blocks. A reader of any single weekly notebook sees the story but not the machinery. This notebook shows and explains every piece in one place.

The code below covers the full pipeline: the raw-file schema and leakage classification, the leakage-safe feature builder and its guards, the deterministic splits, the full-population assembly, the calibration-first evaluation metrics, the baseline and gradient-boosting model factories, the deep tabular benchmarks (TabNet and FT-Transformer), the stacked ensemble, the out-of-time robustness helpers, the SHAP explainability layer, the Black-White gap decomposition, and the fairness-mitigation wrappers.

### Contents

1. [How to read this notebook](#read)
2. [The schema and leakage classification](#schema)
3. [The leakage-safe feature builder](#build)
4. [The leakage guards](#leakage)
5. [Deterministic splits](#splits)
6. [Full-population assembly](#full)
7. [Calibration-first evaluation](#evaluate)
8. [Baseline models](#baselines)
9. [Gradient boosting](#boosting)
10. [Deep tabular benchmarks](#deep)
11. [Stacked ensemble](#ensemble)
12. [Out-of-time robustness](#robustness)
13. [SHAP explainability](#shap)
14. [Black-White gap decomposition](#decomposition)
15. [Fairness mitigation](#mitigation)
16. [The inlined notebook helpers](#helpers)
17. [How it all fits, and the test map](#map)

<a id="read"></a>
## How to read this notebook

The project follows one principle above all: **import, do not re-implement**. The weekly notebooks never rewrite the cleaning, feature engineering, leakage gates, splitting, or scoring logic in a cell. They call the same code the test suite covers, so a notebook cannot quietly diverge from the tested pipeline. That is what makes the honesty checkable by someone who does not trust me.

This reference doubles down on that principle. Every code listing you see below prints the **live source** of the `equibirth` package via `inspect.getsource`. The setup cell imports the package directly from the working-tree `src/` (added to `sys.path` in setup), so what prints is exactly the code the tests run against. If a function changes, this notebook changes with it the next time it is executed: there is no copy to fall out of date.

A few reading notes. Where a piece of logic is a module-level constant (a set of column names, a dictionary) rather than a function or class, `inspect.getsource` cannot fetch it, so those are shown by printing the constant's value directly or by printing the whole module. A handful of listings are display-only: the model factories and the full-population loaders are never *run* here because they would train on or materialize far too much data for a reference. Those are called out explicitly. And the final section is the inlined notebook SETUP helpers, which are not importable and so are pasted (as read-only text) from the generators rather than shown via `inspect`.

First, the setup cell. It imports the fourteen modules, defines `show(obj)` (the thin wrapper around `inspect.getsource`), and defines `load_slice`, which reads a small single-year slice of the cleaned data so the live examples run fast and stay memory-safe.

In [1]:
import sys, inspect, warnings
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))
import numpy as np, polars as pl
warnings.filterwarnings("ignore")

from equibirth.ingest import schema
from equibirth.features import build as fbuild, leakage as fleak, splits as fsplits, full_dataset as ffull
from equibirth.models import evaluate as mev, baselines as mbase, boosting as mboost
from equibirth.models import deep as mdeep, ensemble as mens
from equibirth.eval import robustness as erob
from equibirth.fairness import mitigation as fmit, decomposition as fdecomp
from equibirth.explain import shap_analysis as mshap

def show(obj):
    """Print the LIVE source of a function/class so this notebook cannot drift from the tested code."""
    print(inspect.getsource(obj))

INTERIM = Path("..") / "data" / "interim"
def load_slice(n=200_000, year=2024):
    """A small single-year slice for fast, memory-safe live examples."""
    return pl.scan_parquet(INTERIM / f"natality{year}_clean.parquet").head(n).collect()


In [2]:
print('Modules surfaced in this reference:')
for m in [schema, fbuild, fleak, fsplits, ffull, mev, mbase, mboost, mdeep, mens, erob, mshap, fdecomp, fmit]:
    print(f'  {m.__name__:<40} {m.__doc__.strip().splitlines()[0]}')

Modules surfaced in this reference:
  equibirth.ingest.schema                  Schema, column mapping, and leakage classification for the NBER natality CSV.
  equibirth.features.build                 Build the primary pre-delivery feature matrix (SPEC FR6).
  equibirth.features.leakage               Automated leakage guards (SPEC FR6.1 and FR6.3).
  equibirth.features.splits                Deterministic, stratified train/validation/test splits (SPEC FR5).
  equibirth.features.full_dataset          Full-population assembly for the complete 2018-2024 rebuild (Phase 2 onward).
  equibirth.models.evaluate                Calibration-first evaluation utilities (SPEC FR8).
  equibirth.models.baselines               Baseline models with calibration-first discipline (SPEC FR7, Phase 3).
  equibirth.models.boosting                Gradient-boosted trees with Optuna tuning and post-hoc calibration (SPEC FR7, Phase 4).
  equibirth.models.deep                    Deep tabular benchmarks: TabNet and F

<a id="schema"></a>
## The schema and leakage classification

`equibirth.ingest.schema` is the enforceable core of the project's integrity. It expresses the leakage classification as **sets of raw column names** rather than prose, precisely so that the tests and the automated detector can enumerate columns deterministically instead of matching fuzzy concept descriptions. Here is the whole module.

In [3]:
print(inspect.getsource(schema))

"""Schema, column mapping, and leakage classification for the NBER natality CSV.

All raw column names are the lowercase names served in the NBER per-year CSV (confirmed against the
2023 and 2024 files via the FR1.1 gate; see DECISION_LOG D5). Names are standardized across years by
NBER, but the loader validates presence per year and fails loudly on a mismatch.

The leakage classification is the enforceable core of the project's integrity (SPEC Section 6). It is
expressed as sets of raw column names so tests and the FR6 detector can enumerate columns
deterministically rather than matching concept descriptions.
"""

from __future__ import annotations

# --- Target and its forbidden sibling --------------------------------------------------------------

# Obstetric estimate of gestation in completed weeks. preterm = OBSTETRIC_GESTATION < 37.
OBSTETRIC_GESTATION = "oegest_comb"
# LMP-based gestation. Correlated with the target; EXCLUDED so it cannot encode the outcome.
LMP_GESTATION = "co

The classification splits the birth certificate by one property: **timing**. A field is a fair predictor only if it is known before delivery. The key structures are:

- **`COLUMN_MAP`**: the raw-name to stable-analysis-name mapping, populated only for legitimate pre-delivery and (gated) timing-conditional variables. Leakage variables are deliberately left out of it so they cannot drift into the feature matrix by accident.
- **`LEAKAGE_DETECTOR_VARS`**: fields recorded at or after delivery, or given *because* preterm was anticipated (birthweight, Apgar, NICU, antenatal steroids, delivery route, total prenatal visits, and more). These feed the automated leakage detector: a classifier trained on these alone must not beat the gate.
- **`STRUCTURAL_EXCLUSION_VARS`**: barred by logic rather than by post-delivery leakage. The LMP-based gestation (`combgest`, which would encode the target), plurality (the analysis is singleton-only), and the gestation recodes/flags. These are excluded but kept OUT of the detector because their association is biology or target-encoding, not leakage, and including them causes false-positive gate failures.
- **`IATROGENIC_ENTANGLED_VARS`**: pregnancy conditions that frequently *cause* a preterm delivery via induction (gestational diabetes, gestational hypertension, eclampsia). Their univariate signal is small, but the mechanism is reverse-causal, so they are excluded from the primary model and kept only for post-hoc disparity analysis on the raw frame.
- **`TIMING_CONDITIONAL_VARS`**: kept in the primary model only with an associational caveat and a univariate gate (month prenatal care began, WIC enrollment). Weakly predictive alone, and not reverse-causal at delivery.
- **`MEDICAL_RISK_COMPOSITE_COMPONENTS`**: the exact enumerated members of the medical-risk count feature (pre-pregnancy diabetes, pre-pregnancy hypertension, prior preterm, prior cesarean). Gestational diabetes is deliberately absent because it is a timing leak.
- **`SENTINEL_MAP`**: the per-variable 'unknown / not stated' codes, never a single global value, so a legitimate 9 prenatal visits or parity 9 survives de-sentinelling.
- **`AGE_BRACKET_BREAKS`**: the single source of truth for the maternal-age brackets (the age relationship with preterm is U-shaped), shared so `build.py` and the EDA cannot drift.

In [4]:
print('LEAKAGE_DETECTOR_VARS:', len(schema.LEAKAGE_DETECTOR_VARS), 'columns')
print(sorted(schema.LEAKAGE_DETECTOR_VARS))
print('\nTotal variables that must never enter the primary matrix '
      '(all_excluded_vars):', len(schema.all_excluded_vars()))

LEAKAGE_DETECTOR_VARS: 50 columns
['ab_anti', 'ab_aven1', 'ab_aven6', 'ab_nicu', 'ab_seiz', 'ab_surf', 'apgar10', 'apgar10r', 'apgar5', 'apgar5r', 'attend', 'bfacil', 'bfacil3', 'bwtr12', 'bwtr4', 'ca_anen', 'ca_cchd', 'ca_cdh', 'ca_cleft', 'ca_clpal', 'ca_disor', 'ca_down', 'ca_gast', 'ca_hypo', 'ca_limb', 'ca_mnsb', 'ca_omph', 'dbwt', 'dmeth_rec', 'ip_chlam', 'ip_gon', 'ip_hepb', 'ip_hepc', 'ip_syph', 'ld_anes', 'ld_antb', 'ld_augm', 'ld_chor', 'ld_indl', 'ld_ster', 'me_pres', 'me_rout', 'me_trial', 'no_infec', 'previs', 'previs_rec', 'rdmeth_rec', 'rf_artec', 'rf_fedrg', 'rf_inftr']

Total variables that must never enter the primary matrix (all_excluded_vars): 57


The three helper functions in the module are worth showing on their own: `needed_raw_columns` (the union of every raw column the pipeline retains in the slim interim Parquet), `harmonized_name` (raw name to analysis name, or the raw name if unmapped), and `all_excluded_vars` (leakage-detector plus structural exclusions: everything barred from the primary matrix).

In [5]:
show(schema.needed_raw_columns)

def needed_raw_columns() -> list[str]:
    """Union of every raw column the pipeline keeps in the slim interim Parquet.

    Includes the target, the legitimate/timing/structural features, and the leakage variables (the
    leakage vars are retained ONLY so the Phase 2 leakage-demonstration model and the FR6.1 detector
    can use them; they never enter the primary feature matrix). Extra tobacco recodes are kept.
    """
    extra = {OBSTETRIC_GESTATION, "cig_1", "cig_2", "cig_3"}
    cols = (
        set(COLUMN_MAP)
        | LEAKAGE_DETECTOR_VARS
        | STRUCTURAL_EXCLUSION_VARS
        | TIMING_CONDITIONAL_VARS
        | IATROGENIC_ENTANGLED_VARS   # retained for post-hoc disparity analysis, not as features
        | extra
    )
    return sorted(cols)



In [6]:
show(schema.harmonized_name)

def harmonized_name(raw: str) -> str:
    """Return the analysis name for a raw column, or the raw name if unmapped."""
    return COLUMN_MAP.get(raw, raw)



In [7]:
show(schema.all_excluded_vars)

def all_excluded_vars() -> frozenset[str]:
    """Every variable that must never enter the primary feature matrix."""
    return LEAKAGE_DETECTOR_VARS | STRUCTURAL_EXCLUSION_VARS



Why is the classification code rather than prose? Because a set of column names is enumerable and testable. A test can assert that `dbwt` is in `LEAKAGE_DETECTOR_VARS`, that no engineered feature draws from `all_excluded_vars()`, and that the detector trained on the barred set fails the build if it discriminates. A paragraph describing 'fields known only at delivery' cannot be checked automatically; a `frozenset` can.

<a id="build"></a>
## The leakage-safe feature builder

`equibirth.features.build.build_modeling_frame` turns a cleaned raw frame into the primary pre-delivery feature matrix. It keeps only legitimate pre-conception / early-pregnancy variables plus the gated timing-conditional fields, and engineers a few features on top: the medical-risk count, the U-shaped age bracket, one-hot encodings for the nominal categoricals (race, payment), and a race-by-education interaction. Crucially, **every engineered feature declares the raw columns it is built from**, which is what lets the guard prove nothing leaky was laundered into a composite. The encoding is deliberately simple and transparent: ordinal where genuinely ordered (education), one-hot for nominal categoricals, Y/N flags mapped to 1/0, numerics kept as-is.

In [8]:
show(fbuild.build_modeling_frame)

def build_modeling_frame(
    df: pl.DataFrame, *, include_timing_conditional: bool = True
) -> tuple[pl.DataFrame, pl.Series, dict[str, tuple[str, ...]]]:
    """Return (features, target, component_map) for the primary pre-delivery model.

    ``df`` is a cleaned frame from the ingest layer (raw NBER column names, de-sentinelled, with the
    ``preterm`` target and ``mother_race_ethnicity`` label already present).
    """
    components: dict[str, tuple[str, ...]] = {}
    exprs: list[pl.Expr] = []

    # --- numeric legitimate features ---
    for col in ["mager", "meduc", "bmi", "cig_0", "priorlive", "priordead"]:
        if col in df.columns:
            exprs.append(pl.col(col).cast(pl.Float64, strict=False).alias(col))
            components[col] = (col,)

    # married flag from dmar (1=married, 2=unmarried)
    if "dmar" in df.columns:
        exprs.append((pl.col("dmar") == 1).cast(pl.Int8).alias("married"))
        components["married"] = ("dmar",)

    # Y/N legitimate flags

The little `_yn_to_int` helper is worth its own look: it maps a Y/N string flag to 1/0 and leaves anything else (including the 'U' unknown code, and nulls) as null rather than silently coercing it to 0. That care matters, because a silent coercion to 0 would turn 'unknown' into 'no' and bias every flag toward the negative.

In [9]:
show(fbuild._yn_to_int)

def _yn_to_int(col: str) -> pl.Expr:
    """Map a Y/N string flag to 1/0 (anything else, incl. null, stays null)."""
    return (
        pl.col(col).str.to_uppercase().replace_strict({"Y": 1, "N": 0}, default=None,
                                                       return_dtype=pl.Int8).alias(col)
    )



A live example on a small single-year slice. `build_modeling_frame` returns the feature matrix `X`, the target `y`, and the component map (which raw columns each feature is built from). Note the medical-risk composite is an explicit, enumerated tuple, not an open-ended sum, so no leaky field can hide in it.

In [10]:
df = load_slice()
X, y, comps = fbuild.build_modeling_frame(df)
print('feature matrix shape:', X.shape)
print('first 12 features:', X.columns[:12])
print('medical_risk_count is built from exactly:', comps['medical_risk_count'])

feature matrix shape: (200000, 103)
first 12 features: ['mager', 'meduc', 'bmi', 'cig_0', 'priorlive', 'priordead', 'married', 'rf_pdiab', 'rf_phype', 'rf_ppterm', 'rf_cesar', 'medical_risk_count']
medical_risk_count is built from exactly: ('rf_pdiab', 'rf_phype', 'rf_ppterm', 'rf_cesar')


<a id="leakage"></a>
## The leakage guards

`equibirth.features.leakage` holds the automated guards that keep the honesty claim checkable. `excluded_vars_detector_auc` trains a trivial classifier on the barred post-delivery variables alone: if those fields can predict the target well on their own (AUC above the gate), the exclusion list has a hole or a leaky column slipped in, and the build should fail. It returns a clean 0.5 (no detectable leak) when none of the barred columns are present, which is what happens when it is called on an already-clean feature matrix rather than the raw frame.

In [11]:
show(fleak.excluded_vars_detector_auc)

def excluded_vars_detector_auc(df: pl.DataFrame, target: str = "preterm") -> float:
    """AUC of a classifier trained on the leakage-detector variables alone.

    Returns 0.5 (no discriminability, hence no detectable leak) when none of the leakage-detector
    columns are present, e.g. when called on an already-cleaned feature matrix rather than the raw
    ingested frame.
    """
    valid, y = _target(df, target)
    cols = sorted(schema.LEAKAGE_DETECTOR_VARS)
    if not any(c in valid.columns for c in cols):
        return 0.5
    X = _matrix(valid, cols)
    return _fit_auc(X, y)



`univariate_auc` measures single-feature discrimination. It is used to gate the timing-conditional features we do keep: each must be weak on its own or it is dropped.

In [12]:
show(fleak.univariate_auc)

def univariate_auc(df: pl.DataFrame, col: str, target: str = "preterm") -> float:
    """Single-feature held-out AUC, used to gate timing-conditional features (FR6.3)."""
    valid, y = _target(df, target)
    X = _matrix(valid, [col])
    return _fit_auc(X, y)



`assert_no_component_is_excluded` is the composite guard. Because every engineered feature declares its source columns, this check can prove that none of those sources is a barred variable. `build_modeling_frame` calls it, so a leaky composite cannot be built at all.

In [13]:
show(fleak.assert_no_component_is_excluded)

def assert_no_component_is_excluded(feature_components: dict[str, tuple[str, ...]]) -> None:
    """FR6.2 guard: no engineered feature may be built from an excluded source variable."""
    excluded = schema.all_excluded_vars()
    for feature, components in feature_components.items():
        bad = set(components) & excluded
        if bad:
            raise AssertionError(f"Feature {feature!r} is built from excluded variable(s): {sorted(bad)}")



Live: on the raw single-year slice, the detector should register the leak (the barred fields are present and highly predictive), and a single barred field like birthweight (`dbwt`) should score very high on its own. This is the trap the whole leakage discipline exists to prevent.

In [14]:
slc = load_slice()
print('excluded-vars detector AUC on the raw slice:', round(fleak.excluded_vars_detector_auc(slc), 3))
print('univariate AUC of birthweight (dbwt) alone :', round(fleak.univariate_auc(slc, 'dbwt'), 3))

excluded-vars detector AUC on the raw slice: 0.913
univariate AUC of birthweight (dbwt) alone : 0.878


<a id="splits"></a>
## Deterministic splits

`equibirth.features.splits.assign_splits` produces the deterministic, stratified train/validation/test/held-out partition. The naming is deliberate and avoids a common trap: instead of a single ambiguous 'validation' set (is that for tuning or for final reporting?), the partitions are `held_out_final` (untouched until the very end; all headline metrics and bootstrap CIs use it), `dev_train`, `dev_val` (scores Optuna trials and early stopping), and `dev_test` (calibration and a quick in-development smoke check). Splits are stratified by a composite key (outcome by race/ethnicity by year) so every subgroup is represented in each partition, and are reproducible from the seed.

In [15]:
show(fsplits.assign_splits)

def assign_splits(
    df: pl.DataFrame,
    *,
    stratify_cols: list[str] | None = None,
    held_out_frac: float = 0.4,
    dev_val_frac_of_dev: float = 0.15,
    dev_test_frac_of_dev: float = 0.15,
    seed: int = 42,
) -> pl.Series:
    """Return a ``split`` label Series aligned to ``df`` rows.

    ``held_out_frac`` is a fraction of the WHOLE dataset. ``dev_val_frac_of_dev`` and
    ``dev_test_frac_of_dev`` are fractions of the DEVELOPMENT partition (what remains after holding
    out), not of the whole dataset. Example: held_out_frac=0.4, dev_*=0.15 gives roughly 40% held
    out, and within the remaining 60% dev rows, 15% val and 15% test, i.e. 40 / 42 / 9 / 9 overall.

    Rows in a rare stratum (fewer than 2 members) cannot be stratified; they are assigned to
    ``dev_train`` up front so no row is dropped and the split stays deterministic.
    """
    stratify_cols = stratify_cols or DEFAULT_STRATIFY
    n = df.height
    key = _strata_key(df, stratify_cols)
    labels = np

`stratified_subsample` draws a tractable working sample from the complete ~25M-row population for model development (the full population is too large to hold as a dense matrix), while the full-population outcome audit still runs on every row. Selection is proportional within each stratum with largest-remainder rounding, so the sample's marginal distributions match the population and the total lands exactly on the requested size.

In [16]:
show(fsplits.stratified_subsample)

def stratified_subsample(
    df: pl.DataFrame,
    n: int,
    *,
    stratify_cols: list[str] | None = None,
    seed: int = 42,
) -> pl.Series:
    """Return a boolean mask selecting ~``n`` rows, stratified to preserve the outcome/race mix.

    Used to draw a tractable working sample from the complete ~25M-row population for model
    development, while the full-population outcome audit still runs on every row (SPEC FR5/FR8.1:
    separate the outcome equity audit from the model-performance audit). Selection is proportional
    within each stratum (outcome x race/ethnicity x year by default), so the sample's marginal
    distributions match the population. Deterministic given the seed. If ``n`` is at least the
    population size, every row is selected.
    """
    total = df.height
    if n >= total or total == 0:
        return pl.Series("selected", [True] * total)

    stratify_cols = stratify_cols or DEFAULT_STRATIFY
    key = _strata_key(df, stratify_cols)
    rng = np.random.

Live: assign splits on the slice and look at the partition sizes, then the preterm rate within each partition. Because the split is stratified on the outcome, the rate should be close to constant across partitions, which is the point: no partition is accidentally easier or harder than another.

In [17]:
d = load_slice()
s = fsplits.assign_splits(d)
print('partition sizes:')
print(s.value_counts())
labelled = d.select(pl.col('preterm')).with_columns(s.alias('split'))
rates = (labelled.filter(pl.col('preterm').is_not_null())
         .group_by('split').agg(preterm_rate=pl.col('preterm').mean(), n=pl.len())
         .sort('split'))
print('\npreterm rate by partition (should be nearly constant):')
print(rates)

partition sizes:
shape: (4, 2)
┌────────────────┬───────┐
│ split          ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ held_out_final ┆ 80000 │
│ dev_train      ┆ 84000 │
│ dev_val        ┆ 18000 │
│ dev_test       ┆ 18000 │
└────────────────┴───────┘

preterm rate by partition (should be nearly constant):
shape: (4, 3)
┌────────────────┬──────────────┬───────┐
│ split          ┆ preterm_rate ┆ n     │
│ ---            ┆ ---          ┆ ---   │
│ str            ┆ f64          ┆ u32   │
╞════════════════╪══════════════╪═══════╡
│ dev_test       ┆ 0.091536     ┆ 17982 │
│ dev_train      ┆ 0.091222     ┆ 83927 │
│ dev_val        ┆ 0.090899     ┆ 17987 │
│ held_out_final ┆ 0.091228     ┆ 79932 │
└────────────────┴──────────────┴───────┘


<a id="full"></a>
## Full-population assembly

`equibirth.features.full_dataset` composes the tested ingest and feature layers for the complete 2018-2024 rebuild. It re-implements nothing: it reads the per-year cleaned Parquet files and splits the work exactly as the SPEC prescribes into two separate jobs. `load_full_frame` pools the per-year cleaned files into one frame tagged with `birth_year`.

In [18]:
show(ffull.load_full_frame)

def load_full_frame(interim_dir: str | Path, years: list[int]) -> pl.DataFrame:
    """Pool the per-year cleaned Parquet files into one frame tagged with ``birth_year``.

    Each ``natality{year}_clean.parquet`` is the output of the tested ingest pipeline on the complete
    CSV for that year (singletons, de-sentinelled, target + race label already present).
    """
    interim_dir = Path(interim_dir)
    frames = []
    for year in years:
        path = interim_dir / f"natality{year}_clean.parquet"
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing cleaned file for {year}: {path}. Run scripts/week1_build_full_dataset.py."
            )
        lf = pl.scan_parquet(path)
        if "birth_year" not in lf.collect_schema().names():
            lf = lf.with_columns(pl.lit(year).alias("birth_year"))
        frames.append(lf.collect())
    return pl.concat(frames, how="diagonal_relaxed")



`outcome_audit` runs on EVERY row of the pooled population. Because it is the full census, even the smallest race groups are reliable here, so this is the equity story: overall rate, per-year rate, per-race rate and count, and the NH Black minus NH White gap in percentage points. It is strictly descriptive and associational: race is a marker of exposure to racism and structural barriers, not a biological cause.

In [19]:
show(ffull.outcome_audit)

def outcome_audit(df: pl.DataFrame) -> dict:
    """Full-population outcome/equity audit (every row, so small groups stay reliable).

    Returns overall rate, per-year rate, per-race rate + count, and the NH Black minus NH White gap
    in percentage points. All descriptive and associational: race is a marker of exposure to racism,
    not a biological cause.
    """
    valid = df.filter(pl.col("preterm").is_not_null())
    n = valid.height
    overall = float(valid["preterm"].mean()) if n else float("nan")

    by_race = (
        valid.filter(pl.col("mother_race_ethnicity").is_not_null())
        .group_by("mother_race_ethnicity")
        .agg(rate=pl.col("preterm").mean(), n=pl.len())
        .sort("rate")
    )
    by_year = (
        valid.group_by("birth_year").agg(rate=pl.col("preterm").mean(), n=pl.len()).sort("birth_year")
        if "birth_year" in valid.columns
        else pl.DataFrame({"birth_year": [], "rate": [], "n": []})
    )

    rate_by_race = [
        {"mother_r

`assemble_modeling` is the other half: it draws the deterministic stratified working sample, builds the leakage-safe matrix via the tested `build_modeling_frame`, and assigns the same deterministic stratified splits, returning the bundle the models consume. This separation, the full-census **outcome audit** kept apart from the sampled **modeling** job, is the key design point: how unequal the world is (measured on everyone) is a different question from how well we can predict (measured on a tractable sample with an untouched held-out fold).

These functions are shown for reference only and are **not run here**: `load_full_frame` and `assemble_modeling` would read and materialize the full ~24.9M-row population, which belongs in a script, not a reference cell.

In [20]:
show(ffull.assemble_modeling)

def assemble_modeling(
    df: pl.DataFrame,
    *,
    working_size: int,
    held_out_frac: float = 0.5,
    seed: int = 42,
) -> dict:
    """Draw a stratified working sample and build the leakage-safe modeling bundle.

    ``working_size`` bounds the number of rows carried into training/evaluation (the full population
    is too large to hold as a dense matrix). Splits are stratified and deterministic; the returned
    ``held_out_final`` partition is what final metrics and bootstrap CIs are computed on.
    """
    mask = splits.stratified_subsample(df, working_size, seed=seed)
    work = df.filter(mask)

    X, y, components = build_modeling_frame(work)
    meta = work.select(
        [c for c in ["birth_year", "mother_race_ethnicity", "meduc", "pay"] if c in work.columns]
    )
    split = splits.assign_splits(work, held_out_frac=held_out_frac, seed=seed)
    return {"X": X, "y": y, "meta": meta, "split": split, "components": components}



<a id="evaluate"></a>
## Calibration-first evaluation

`equibirth.models.evaluate` is where the calibration-first discipline lives. Because the outcome is rare (~9%) and the whole fairness argument rests on calibration-by-group being trustworthy, the project leads with rate-aware metrics and calibration, never accuracy. Here are the metric functions, each with a note on why it earns its place.

`auc_roc` is ROC AUC. It is base-rate invariant (its floor is 0.5 regardless of prevalence), which is exactly why it is the fair thing to compare ACROSS race groups whose prevalence differs.

In [21]:
show(mev.auc_roc)

def auc_roc(y: np.ndarray, p: np.ndarray) -> float:
    """ROC AUC. Base-rate invariant, so it is the metric compared across race groups."""
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, p))



`auc_pr` is average precision (area under the precision-recall curve). Its floor equals the prevalence, so under this much imbalance a useless model cannot hide behind a high number: that makes it the honest primary metric.

In [22]:
show(mev.auc_pr)

def auc_pr(y: np.ndarray, p: np.ndarray) -> float:
    """Average precision (area under the precision-recall curve). The primary metric under imbalance."""
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(average_precision_score(y, p))



`brier` is the Brier score, a proper scoring rule (the mean squared error of the predicted probability). Lower is better, and it rewards being both discriminating and calibrated.

In [23]:
show(mev.brier)

def brier(y: np.ndarray, p: np.ndarray) -> float:
    """Brier score (mean squared error of the probability). Lower is better; a proper scoring rule."""
    return float(brier_score_loss(np.asarray(y), np.asarray(p)))



`ece` is the expected calibration error over equal-width probability bins: the count-weighted mean gap between predicted and observed rate. It is the headline calibration number, the one that answers 'when the model says 10%, is the real rate about 10%?'.

In [24]:
show(mev.ece)

def ece(y: np.ndarray, p: np.ndarray, n_bins: int = 10, *, min_bin_size: int = 1) -> float:
    """Expected calibration error over ``n_bins`` equal-width probability bins."""
    return _calibration_error(y, p, _bin_edges_uniform(n_bins), worst=False, min_bin_size=min_bin_size)



`ece_quantile` is the same idea over equal-MASS bins. It is the stricter summary when predictions pile up near the base rate, because equal-width bins can leave the crowded low-risk region under-resolved.

In [25]:
show(mev.ece_quantile)

def ece_quantile(y: np.ndarray, p: np.ndarray, n_bins: int = 10, *, min_bin_size: int = 1) -> float:
    """ECE over equal-MASS bins. Stricter when predictions pile up near the base rate."""
    return _calibration_error(y, p, _bin_edges_quantile(np.asarray(p, dtype=float), n_bins),
                              worst=False, min_bin_size=min_bin_size)



`mce` is the maximum calibration error, with a minimum-bin-size guard so a single sparse high-probability bin (a handful of rows) cannot inflate it into a binning artifact.

In [26]:
show(mev.mce)

def mce(y: np.ndarray, p: np.ndarray, n_bins: int = 10, *, min_bin_size: int = 30) -> float:
    """Maximum calibration error, ignoring bins with fewer than ``min_bin_size`` rows.

    The guard stops a single sparse bin (a handful of high-probability rows) from inflating MCE into
    a binning artifact, which is what the raw MCE did in the sample-era analysis (DECISION_LOG D31).
    """
    return _calibration_error(y, p, _bin_edges_uniform(n_bins), worst=True, min_bin_size=min_bin_size)



`calibration_curve` returns the per-bin mean-predicted, observed-rate, and count arrays used to draw a reliability diagram.

In [27]:
show(mev.calibration_curve)

def calibration_curve(y: np.ndarray, p: np.ndarray, n_bins: int = 10):
    """Return (mean_predicted, observed_rate, counts) per equal-width bin for a reliability diagram."""
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    edges = _bin_edges_uniform(n_bins)
    idx = np.clip(np.digitize(p, edges[1:-1], right=False), 0, n_bins - 1)
    mean_pred = np.full(n_bins, np.nan)
    observed = np.full(n_bins, np.nan)
    counts = np.zeros(n_bins, dtype=int)
    for b in range(n_bins):
        m = idx == b
        counts[b] = int(m.sum())
        if counts[b]:
            mean_pred[b] = p[m].mean()
            observed[b] = y[m].mean()
    return mean_pred, observed, counts



`fraction_near_base_rate` is the honest qualification in code: the share of predictions within a small window of the base rate. A high value means the model rarely moves far from the prevalence, so a low ECE reflects aggregate calibration rather than individual risk stratification. Both things can be true at once, and this function makes the distinction measurable.

In [28]:
show(mev.fraction_near_base_rate)

def fraction_near_base_rate(p: np.ndarray, base_rate: float, window: float = 0.05) -> float:
    """Share of predictions within ``window`` of the base rate.

    A high value means the model rarely moves far from the prevalence, so low ECE reflects aggregate
    calibration rather than individual risk stratification (the honest qualification from D31).
    """
    p = np.asarray(p, dtype=float)
    return float(np.mean(np.abs(p - base_rate) <= window)) if len(p) else float("nan")



`bootstrap_ci` gives a stratified bootstrap confidence interval for any metric. Resampling is stratified by outcome class so each resample keeps the ~9% prevalence, and degenerate single-class resamples are skipped.

In [29]:
show(mev.bootstrap_ci)

def bootstrap_ci(metric_fn, y: np.ndarray, p: np.ndarray, *, n: int = 2000, seed: int = 42,
                 alpha: float = 0.05):
    """Stratified bootstrap CI for a metric. Returns (point, lo, hi).

    Resampling is stratified by outcome class so each resample keeps the prevalence, which matters at
    the ~9% base rate. Degenerate resamples (a single class) are skipped.
    """
    y, p = np.asarray(y), np.asarray(p)
    point = metric_fn(y, p)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    if len(pos) == 0 or len(neg) == 0:
        return float(point), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n):
        bi = np.concatenate([rng.choice(pos, len(pos), replace=True),
                             rng.choice(neg, len(neg), replace=True)])
        val = metric_fn(y[bi], p[bi])
        if val == val:  # skip NaN
            stats.append(val)
    if not stats:
        return float(point), float("nan"), floa

`stratified_report` is the per-group reporting with tiered suppression: groups with too few positives are suppressed, mid-sized groups are reported but flagged wide-CI, and only well-populated groups are reported as reliable. Small race groups are flagged, never silently dropped.

In [30]:
show(mev.stratified_report)

def stratified_report(y: np.ndarray, p: np.ndarray, groups: np.ndarray, *,
                      suppress_floor: int = 50, full_ci_floor: int = 1000) -> list[GroupResult]:
    """Per-group metrics with tiered suppression (SPEC FR8.1).

    - fewer than ``suppress_floor`` positives (or negatives): suppressed (privacy + noise);
    - ``suppress_floor`` up to ``full_ci_floor`` positives: reported but flagged ``wide_ci``;
    - at least ``full_ci_floor`` positives: ``ok``.

    Small race groups are flagged, never silently dropped.
    """
    y, p, groups = np.asarray(y), np.asarray(p), np.asarray(groups)
    out: list[GroupResult] = []
    for g in sorted({str(v) for v in groups if v is not None and str(v) != "nan"}):
        m = groups == g
        yy, pp = y[m], p[m]
        n_g = int(m.sum())
        pos = int((yy == 1).sum())
        neg = int((yy == 0).sum())
        rate = float(yy.mean()) if n_g else float("nan")
        if pos < suppress_floor or neg < suppress_floor:
           

A tiny live example so the metrics are concrete. On a toy label/probability pair, `auc_roc`, `auc_pr`, and `ece` return real numbers.

In [31]:
y = np.array([0, 0, 1, 0, 1, 1])
p = np.array([.1, .2, .7, .3, .6, .9])
print('auc_roc:', round(mev.auc_roc(y, p), 3))
print('auc_pr :', round(mev.auc_pr(y, p), 3))
print('ece    :', round(mev.ece(y, p), 3))

auc_roc: 1.0
auc_pr : 1.0
ece    : 0.233


<a id="baselines"></a>
## Baseline models

`equibirth.models.baselines` builds the three Week 3 baselines: logistic regression (the calibration reference and honest floor), a decision tree, and a random forest. `_make_estimators` constructs the three pipelines, each with a median imputer (and a scaler for logistic regression) so callers can pass a raw matrix with missing values.

In [32]:
show(mbase._make_estimators)

def _make_estimators(seed: int) -> dict:
    """Construct the three baseline pipelines with balanced class weights."""
    return {
        "logistic_regression": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed)),
        ]),
        "decision_tree": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("clf", DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                           class_weight="balanced", random_state=seed)),
        ]),
        "random_forest": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            # max_depth + a larger min_samples_leaf bound tree size so the forest stays memory-safe
            # on ~1M rows (unbounded trees on 1M rows exhaust RAM), with negligible metric cost here.
            # n_jobs is bounded (not -1): each 

`fit_predict_raw_and_calibrated` fits each baseline once and returns both the raw and the isotonic-calibrated held-out probabilities. Fitting once (rather than twice) halves the memory and time, which matters when the forest trains on roughly a million rows.

Two design choices are worth stating. First, **class weights, not SMOTE**: the imbalance is handled with balanced class weights rather than synthetic resampling, because resampling distorts the base rate and therefore corrupts the very calibration the fairness audit depends on. Second, **isotonic recalibration on a SEPARATE fold**: the calibrator is fit on a held-aside fold the base model never trained on, so the calibrated probabilities are honest out-of-sample and calibration-by-group stays trustworthy.

These are shown for reference only and are **not run here**: fitting the baselines is a training job, not a reference cell.

In [33]:
show(mbase.fit_predict_raw_and_calibrated)

def fit_predict_raw_and_calibrated(X: np.ndarray, y: np.ndarray, tr: np.ndarray, val: np.ndarray,
                                   ho: np.ndarray, *, seed: int = 42):
    """Fit each baseline ONCE and return both raw and calibrated held-out probabilities.

    Returns ``(raw, calibrated)``, each a dict keyed by model name. Fitting once (rather than the two
    separate ``fit_predict`` calls that raw-vs-calibrated comparison would otherwise need) halves the
    memory and time, which matters when the forest is trained on ~1M rows.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    raw: dict[str, np.ndarray] = {}
    calibrated: dict[str, np.ndarray] = {}
    can_calibrate = val is not None and np.asarray(val).sum() > 0 and len(np.unique(y[val])) > 1
    for name, est in _make_estimators(seed).items():
        est.fit(X[tr], y[tr])
        raw[name] = est.predict_proba(X[ho])[:, 1]
        if can_calibrate:
            cal = _prefit_calibrator(est)
            cal.fit

<a id="boosting"></a>
## Gradient boosting

`equibirth.models.boosting` holds the Week 4 gradient-boosting logic: XGBoost and LightGBM, the expected top performers on this tabular data. Both handle missing values natively (no imputer) and use `scale_pos_weight` for the imbalance rather than resampling, again to protect calibration. `_make_estimator` constructs an unfitted booster of the requested family with sensible fixed defaults.

In [34]:
show(mboost._make_estimator)

def _make_estimator(name: str, params: dict, spw: float, seed: int):
    """Construct an unfitted booster of the given family with sensible fixed defaults."""
    name = name.lower()
    if name == "xgboost":
        from xgboost import XGBClassifier

        return XGBClassifier(
            n_estimators=params.get("n_estimators", 400),
            learning_rate=params.get("learning_rate", 0.05),
            max_depth=params.get("max_depth", 6),
            subsample=params.get("subsample", 0.9),
            colsample_bytree=params.get("colsample_bytree", 0.9),
            min_child_weight=params.get("min_child_weight", 1.0),
            reg_lambda=params.get("reg_lambda", 1.0),
            scale_pos_weight=spw,
            tree_method="hist",
            eval_metric="aucpr",
            n_jobs=-1,
            random_state=seed,
        )
    if name == "lightgbm":
        from lightgbm import LGBMClassifier

        return LGBMClassifier(
            n_estimators=params.get("n_estima

`scale_pos_weight` is the standard imbalance weight for boosters: the negative/positive ratio (1.0 when there are no positives).

In [35]:
show(mboost.scale_pos_weight)

def scale_pos_weight(y: np.ndarray) -> float:
    """Return negative/positive ratio for class-weighted boosting. 1.0 if no positives."""
    y = np.asarray(y)
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    return float(neg / pos) if pos else 1.0



`tune` runs an Optuna study that maximizes AUC-PR (the primary metric) on the dev validation fold only, so the held-out set is never touched during tuning. A trial cap and a wall-clock timeout keep it safe in CI and on a laptop.

In [36]:
show(mboost.tune)

def tune(name: str, X: np.ndarray, y: np.ndarray, tr: np.ndarray, val: np.ndarray, *,
         n_trials: int = 50, timeout: int | None = 3600, seed: int = 42) -> dict:
    """Optuna-tune a booster on ``tr``, scoring AUC-PR on ``val``. Returns the best params dict.

    Tuning uses the dev validation fold only; the held-out set is never touched here (no tuning
    leakage). ``timeout`` (seconds) and ``n_trials`` bound the search.
    """
    import optuna
    from sklearn.metrics import average_precision_score

    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    spw = scale_pos_weight(y[tr])
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = _search_space(name, trial)
        est = _make_estimator(name, params, spw, seed)
        est.fit(X[tr], y[tr])
        p = est.predict_proba(X[val])[:, 1]
        return average_precision_score(y[val], p)

    study = optuna.create_study(direction="maximize",
                               

`fit_calibrated` fits a booster on the training fold and isotonic-recalibrates it on a separate fold, returning the fitted calibrated estimator (the fairness script needs it to score several partitions).

These are shown for reference only and are **not run here**: tuning and fitting boosters is a training job.

In [37]:
show(mboost.fit_calibrated)

def fit_calibrated(name: str, X: np.ndarray, y: np.ndarray, tr: np.ndarray, cal: np.ndarray,
                   params: dict, *, seed: int = 42):
    """Fit a booster on ``tr`` and isotonic-recalibrate on ``cal``. Returns the calibrated estimator."""
    from sklearn.calibration import CalibratedClassifierCV

    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    est = _make_estimator(name, params, scale_pos_weight(y[tr]), seed)
    est.fit(X[tr], y[tr])
    try:
        from sklearn.frozen import FrozenEstimator

        calibrated = CalibratedClassifierCV(FrozenEstimator(est), method="isotonic")
    except ImportError:  # pragma: no cover - older sklearn
        calibrated = CalibratedClassifierCV(est, method="isotonic", cv="prefit")
    calibrated.fit(X[cal], y[cal])
    return calibrated



<a id="deep"></a>
## Deep tabular benchmarks

`equibirth.models.deep` provides TabNet (pytorch-tabnet) and FT-Transformer (rtdl) as honest benchmarks behind the same contract as the boosters (D41). The signal in this administrative data is too weak for neural nets to exploit (max |corr| ~0.13, prevalence ~8.6%), so trees are expected to win and did win (F20). The real deliverable is TabNet's attention-based feature masks, which give an independent importance ranking without needing TreeSHAP.

Unlike boosters, neural nets cannot handle NaN, so `_prep` median-imputes (fit on train) and standardizes. The rest follows the same discipline: fit on train, early-stop on val, isotonic-recalibrate on val, report on held-out.

In [38]:
show(mdeep.fit)

def fit(name: str, X: np.ndarray, y: np.ndarray, tr: np.ndarray, val: np.ndarray, *,
        max_epochs: int = 20, seed: int = 42):
    """Fit a deep tabular model on ``tr`` (early-stopping on ``val``).

    Returns ``(est, Xp)``: the fitted estimator plus the imputed+standardized full matrix
    (prep fit on train only), so callers can score any partition once via ``predict`` without
    refitting. ``fit_predict`` below composes this with isotonic calibration.
    """
    name = name.lower()
    if name not in VALID:
        raise ValueError(f"Unknown deep model {name!r}; expected one of {VALID}.")
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y).astype(int)
    tr = np.asarray(tr); val = np.asarray(val)
    _set_torch_determinism(seed)
    np.random.seed(seed)

    Xp = _prep(X, y, tr)
    if name == "tabnet":
        est = _fit_tabnet(Xp, y, tr, val, max_epochs=max_epochs, seed=seed)
    else:
        pos = max(int(y[tr].sum()), 1)
        spw = float((len(y[tr]) - pos)

`predict` scores a partition using the fitted estimator and the already-imputed matrix.

In [39]:
show(mdeep.predict)

def predict(est, Xp: np.ndarray, mask: np.ndarray = None) -> np.ndarray:
    """Raw (uncalibrated) positive-class probabilities on ``Xp[mask]`` (all rows if None)."""
    Z = Xp if mask is None else Xp[np.asarray(mask)]
    return est.predict_proba(Z)[:, 1]



`fit_predict` composes fitting, calibration, and scoring in one call, matching the contract of the boosting module.

In [40]:
show(mdeep.fit_predict)

def fit_predict(name: str, X: np.ndarray, y: np.ndarray, tr: np.ndarray, val: np.ndarray,
                ho: np.ndarray, *, max_epochs: int = 20, seed: int = 42,
                calibrate: bool = True) -> np.ndarray:
    """Fit a deep tabular model on ``tr`` (early-stop on ``val``), isotonic-recalibrate on
    ``val``, and return held-out probabilities on ``ho``."""
    y = np.asarray(y).astype(int)
    val = np.asarray(val); ho = np.asarray(ho)
    est, Xp = fit(name, X, y, tr, val, max_epochs=max_epochs, seed=seed)
    raw_val = predict(est, Xp, val)
    raw_ho = predict(est, Xp, ho)

    if not calibrate or len(np.unique(y[val])) < 2:
        return raw_ho
    from sklearn.isotonic import IsotonicRegression

    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(raw_val, y[val])
    return np.asarray(iso.predict(raw_ho))



`tabnet_feature_masks` extracts the global feature importance from TabNet's built-in explain mechanism. The resulting ranking is associational (same caveats as SHAP) and is checked against the leakage taxonomy before reporting.

In [41]:
show(mdeep.tabnet_feature_masks)

def tabnet_feature_masks(est, Xp: np.ndarray, feature_names: list) -> list:
    """Global TabNet importance from its built-in explain masks (associational, leakage-check
    before reporting). Returns [(feature, weight)] descending."""
    imp = np.asarray(est.feature_importances_, dtype=float)
    order = np.argsort(imp)[::-1]
    return [(feature_names[i], float(imp[i])) for i in order]



<a id="ensemble"></a>
## Stacked ensemble

`equibirth.models.ensemble` implements the cross-fitted logistic meta-learner (roadmap item 2). It stacks base-model probabilities in logit space and uses stratified K-fold cross-fitting to prevent meta-leakage. The stack is deliberately simple: with base models this correlated and signal this weak, anything fancier overfits.

`oof_stack` fits the cross-fitted stack and returns the meta-model plus out-of-fold probabilities.

In [42]:
show(mens.oof_stack)

def oof_stack(base: dict, y: np.ndarray, *, n_folds: int = 5, seed: int = 42):
    """Cross-fitted logistic stack.

    ``base``: {model_name: probs} on a COMMON row set (e.g. dev_val). Returns
    ``(meta_model, oof_probs)`` where ``oof_probs[i]`` comes from the fold whose training
    excluded row i, and ``meta_model`` is refit on all rows for downstream scoring.
    """
    if len(base) < 2:
        raise ValueError("stacking needs at least two base models")
    y = np.asarray(y).astype(int)
    names, Z = _design(base)
    oof = np.zeros(len(y), dtype=float)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for tr_idx, te_idx in skf.split(Z, y):
        m = LogisticRegression(max_iter=2000, random_state=seed).fit(Z[tr_idx], y[tr_idx])
        oof[te_idx] = m.predict_proba(Z[te_idx])[:, 1]
    meta = LogisticRegression(max_iter=2000, random_state=seed).fit(Z, y)
    meta.feature_names_ = names  # so weights are reportable by name
    return meta, oof



`predict_stack` scores a new partition using the fitted meta-model and the base probabilities from that partition.

In [43]:
show(mens.predict_stack)

def predict_stack(meta, base: dict) -> np.ndarray:
    """Score the fitted stack on a new partition's base probabilities."""
    names, Z = _design(base)
    if getattr(meta, "feature_names_", names) != names:
        raise ValueError(f"base model set mismatch: expected {meta.feature_names_}, got {names}")
    return meta.predict_proba(Z)[:, 1]



<a id="robustness"></a>
## Out-of-time robustness

`equibirth.eval.robustness` answers the deployment question a random split cannot: does performance and calibration survive a regime change? Era boundaries are fixed by the pandemic timeline (pre-COVID 2018-2019, COVID 2020-2021, recent 2022-2024), not tuned. A model trained on pre-COVID births is tested on COVID-era and recent-era births.

`out_of_time_masks` returns disjoint boolean masks keyed by era.

In [44]:
show(erob.out_of_time_masks)

def out_of_time_masks(years: np.ndarray) -> dict:
    """Boolean era masks keyed on birth year. Disjoint by construction."""
    years = np.asarray(years)
    return {
        "train_precovid": np.isin(years, PRECOVID_YEARS),
        "test_covid": np.isin(years, COVID_YEARS),
        "test_recent": np.isin(years, RECENT_YEARS),
    }



`yearly_drift` computes per-year n, prevalence, AUC-ROC, AUC-PR, and ECE so drift can be read as a time series rather than a single pre/post comparison.

In [45]:
show(erob.yearly_drift)

def yearly_drift(y: np.ndarray, p: np.ndarray, years: np.ndarray) -> list:
    """Per-year n, prevalence, AUC-ROC, AUC-PR, ECE. Degenerate years get NaN AUCs."""
    from equibirth.models import evaluate as ev

    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    years = np.asarray(years)
    rows = []
    for yr in sorted(np.unique(years).tolist()):
        m = years == yr
        yy, pp = y[m], p[m]
        degenerate = len(np.unique(yy)) < 2
        rows.append({
            "year": int(yr), "n": int(m.sum()), "prevalence": float(yy.mean()),
            "auc_roc": float("nan") if degenerate else ev.auc_roc(yy, pp),
            "auc_pr": float("nan") if degenerate else ev.auc_pr(yy, pp),
            "ece": ev.ece(yy, pp),
        })
    return rows



<a id="shap"></a>
## SHAP explainability

`equibirth.explain.shap_analysis` is the explainability layer Week 7 leans on (SPEC FR10). It turns the fitted tree model into a global feature-importance ranking and, more importantly, race-stratified rankings: if the top predictors differ across groups (say prior preterm birth dominating for one group while access factors dominate for another), that difference is itself a finding about where risk concentrates. Every reading here is associational, never causal.

`shap_values_for` returns the `(n_samples, n_features)` SHAP matrix for a fitted tree model. It normalizes across SHAP API versions: the older per-class list and the newer 3-D `(samples, features, classes)` array both collapse to the positive-class contributions.

In [46]:
show(mshap.shap_values_for)

def shap_values_for(model, X: np.ndarray) -> np.ndarray:
    """Return a (n_samples, n_features) SHAP value matrix for a fitted tree model."""
    import shap

    explainer = shap.TreeExplainer(model)
    vals = explainer.shap_values(X)
    if isinstance(vals, list):           # older API: per-class list
        vals = vals[1] if len(vals) > 1 else vals[0]
    vals = np.asarray(vals)
    if vals.ndim == 3:                   # newer API: (n_samples, n_features, n_classes)
        vals = vals[:, :, -1]            # positive-class contributions
    return vals



`mean_abs_importance` ranks features by mean absolute SHAP value (descending). Mean |SHAP| is the standard global-importance summary: it measures how much each feature moves the prediction on average, regardless of direction.

In [47]:
show(mshap.mean_abs_importance)

def mean_abs_importance(shap_vals: np.ndarray, feature_names: list[str]) -> list[tuple[str, float]]:
    """Rank features by mean absolute SHAP value (descending)."""
    imp = np.abs(shap_vals).mean(axis=0)
    order = np.argsort(imp)[::-1]
    return [(feature_names[i], float(imp[i])) for i in order]



`top_features_by_group` computes the mean-|SHAP| ranking WITHIN each requested race group, returning the top-N per group. This is the mechanism behind the stratified-importance finding: empty groups are skipped rather than erroring, so small groups do not break the call.

In [48]:
show(mshap.top_features_by_group)

def top_features_by_group(
    shap_vals: np.ndarray, feature_names: list[str], groups: np.ndarray, *,
    which: list[str], top_n: int = 8,
) -> dict[str, list[tuple[str, float]]]:
    """Return the top-N features (by mean |SHAP|) within each requested group."""
    out: dict[str, list[tuple[str, float]]] = {}
    for g in which:
        m = groups == g
        if m.sum() == 0:
            continue
        out[g] = mean_abs_importance(shap_vals[m], feature_names)[:top_n]
    return out



A tiny live example. On a toy two-feature problem where a random forest learns that the first feature carries the signal, `mean_abs_importance` should rank it first. This runs fast and needs no fitted project model.

In [49]:
from sklearn.ensemble import RandomForestClassifier
rng = np.random.default_rng(0)
Xt = rng.normal(size=(300, 2))
yt = (Xt[:, 0] > 0).astype(int)
rf = RandomForestClassifier(n_estimators=40, random_state=0).fit(Xt, yt)
sv = mshap.shap_values_for(rf, Xt)
print('ranking:', mshap.mean_abs_importance(sv, ['signal_x0', 'noise_x1']))

ranking: [('signal_x0', 0.4822931218071239), ('noise_x1', 0.013380259701297949)]


<a id="decomposition"></a>
## Black-White gap decomposition

`equibirth.fairness.decomposition` answers research question 4 (SPEC FR10): how much of the observed Black-White preterm gap is statistically associated with measured covariates, and how much is left unexplained? It uses **g-computation (standardization)** over a logistic model: fit preterm on race plus measured covariates, then compare the counterfactual mean risk if everyone were assigned the focal race versus the reference race, holding each person's covariates fixed. The difference is the covariate-adjusted (unexplained) gap; the raw gap minus that adjusted gap is the share associated with the measured covariates.

The heavy caveat is stated in the module docstring and must travel with any use: the covariates (education, payment, WIC) are themselves products of structural racism, so the explained share is inflated and the unexplained residual is a LOWER bound on structural contributions. This is association, not proof of mechanism.

`Decomposition` is the result dataclass: raw gap, adjusted gap, explained absolute and percentage, and the two group sample sizes.

In [50]:
show(fdecomp.Decomposition)

@dataclass
class Decomposition:
    raw_gap: float
    adjusted_gap: float
    explained_abs: float
    explained_pct: float
    n_focal: int
    n_ref: int



`decompose_gap` does the work. It takes the measured covariates (explicitly NOT including race), the target, and the two boolean group masks, median-imputes and standardizes the covariates, fits the logistic model with race plus covariates, and standardizes over the COMBINED covariate distribution so the adjusted gap is not confounded by differing covariate distributions between the two groups.

In [51]:
show(fdecomp.decompose_gap)

def decompose_gap(
    X_cov: np.ndarray, y: np.ndarray, is_focal: np.ndarray, is_ref: np.ndarray, *, seed: int = 42,
) -> Decomposition:
    """g-computation decomposition.

    ``X_cov`` are the measured covariates (NOT including race). ``is_focal`` / ``is_ref`` are boolean
    group masks (e.g. NH Black, NH White). Returns raw vs adjusted gap and the explained share.
    """
    mask = is_focal | is_ref
    Xc = X_cov[mask]
    yy = y[mask]
    race = is_focal[mask].astype(float)  # 1 = focal, 0 = reference

    raw_gap = float(yy[race == 1].mean() - yy[race == 0].mean())

    imp = SimpleImputer(strategy="median").fit(Xc)
    Xi = imp.transform(Xc)
    sc = StandardScaler().fit(Xi)
    Xs = sc.transform(Xi)
    design = np.column_stack([race, Xs])

    model = LogisticRegression(max_iter=2000, random_state=seed).fit(design, yy)

    # counterfactual standardization over the COMBINED covariate distribution
    as_focal = np.column_stack([np.ones(len(Xs)), Xs])
    as_ref = np.column

A live example. Construct a case where a single covariate fully drives the gap (the focal group has a higher covariate value, and the covariate raises risk identically for both groups). The decomposition should attribute nearly all of the raw gap to the covariate, leaving an adjusted (unexplained) gap near zero.

In [52]:
rng = np.random.default_rng(1)
n = 4000
focal = np.array([True] * (n // 2) + [False] * (n // 2))
cov = np.where(focal, rng.normal(1.0, 1, n), rng.normal(-1.0, 1, n))
risk = 1 / (1 + np.exp(-(cov)))
y = (rng.uniform(size=n) < risk).astype(int)
res = fdecomp.decompose_gap(cov.reshape(-1, 1), y, focal, ~focal)
print(f'raw gap      : {res.raw_gap:.4f}')
print(f'adjusted gap : {res.adjusted_gap:.4f}')
print(f'explained pct: {res.explained_pct:.1%}')

raw gap      : 0.3905
adjusted gap : -0.0123
explained pct: 103.1%


<a id="mitigation"></a>
## Fairness mitigation

`equibirth.fairness.mitigation` holds the bias-mitigation experiments (SPEC FR9.2). The module reports the accuracy-fairness tradeoff of each approach; nothing here decides which mitigation to adopt (that is a value judgment, examined in F22/D43).

`expgrad_predictions` wraps Fairlearn's ExponentiatedGradient reductions. It fits a logistic base learner under a TPR-parity or equalized-odds constraint and returns both baseline and mitigated hard predictions, so callers can price the fairness/accuracy tradeoff directly.

In [53]:
show(fmit.expgrad_predictions)

def expgrad_predictions(X_fit: np.ndarray, y_fit: np.ndarray, g_fit: np.ndarray,
                        X_score: np.ndarray, g_score: np.ndarray, *,
                        constraint: str = "equalized_odds", seed: int = 42,
                        max_iter: int = 30):
    """In-processing mitigation via Fairlearn ExponentiatedGradient reductions.

    Fits a logistic base learner under a fairness constraint ("equalized_odds" or
    "true_positive_rate_parity" for equal opportunity). Returns ``(baseline_pred, mitigated_pred)``
    hard 0/1 predictions on the score partition, so callers can price the tradeoff. The reduction
    yields a randomized classifier; we report its realized decisions (seeded, reproducible).
    Logistic base learner keeps the reduction convex and fast; the point is the constraint's
    cost, not squeezing the last AUC point.

    ``g_score`` is accepted for a symmetric fit/score signature but is intentionally unused: the
    fitted ExponentiatedGradient randomi

`threshold_optimizer_predictions` wraps Fairlearn's ThresholdOptimizer: it learns group-specific thresholds on precomputed scores to equalize TPR across groups. This IS disparate treatment (different decision rules by race) and is reported analytically, never deployed.

In [54]:
show(fmit.threshold_optimizer_predictions)

def threshold_optimizer_predictions(y_fit: np.ndarray, p_fit: np.ndarray, g_fit: np.ndarray,
                                    p_score: np.ndarray, g_score: np.ndarray, *,
                                    constraint: str = "true_positive_rate_parity",
                                    seed: int = 42) -> np.ndarray:
    """Post-processing mitigation via Fairlearn ThresholdOptimizer on precomputed scores.

    Learns group-specific thresholds on the fit partition and applies them to the score
    partition. This IS disparate treatment (different rules by group); it is reported as an
    analytical option and never exposed in any deployed artifact.
    """
    from fairlearn.postprocessing import ThresholdOptimizer
    from sklearn.base import BaseEstimator, ClassifierMixin

    class _Precomputed(BaseEstimator, ClassifierMixin):
        """Identity estimator: 'features' are the precomputed probability column."""
        def fit(self, X, y):
            self.classes_ = np.array([0,

<a id="helpers"></a>
## The inlined notebook helpers

Not all of the weekly-notebook logic lives in the package. Some small helpers are defined inline in each weekly notebook's SETUP block: the shared plotting style, the full-data loader, and two pandas EDA workhorses. Because they live in the generators rather than in `src/equibirth/`, they are not importable and `inspect.getsource` cannot reach them, so the cells below are **pasted copies of those SETUP helpers**, documented here in one place. To keep this reference safe to execute, each block is stored as a string and printed rather than run (in particular `load_years` is never called on all years, which would materialize the full ~24.9M-row frame).

First, the shared house style, from the Week 2 generator's SETUP: the Okabe-Ito `PALETTE`, the fixed `RACE_ORDER`, the matplotlib `rcParams` block, and the `style(ax, ...)` axis helper. The same palette and style run through every weekly notebook so the whole project reads as one piece.

In [55]:
print('# The same Okabe-Ito palette and house style shared across every weekly notebook.\nPALETTE = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",\n           "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#999999",\n           "ink": "#1a1a2e"}\nRACE_ORDER = ["NH White", "NH Asian", "Hispanic", "NH Multiracial", "NH NHOPI", "NH AIAN", "NH Black"]\nRACE_COLOR = {r: PALETTE["grey"] for r in RACE_ORDER}\nRACE_COLOR.update({"NH White": PALETTE["blue"], "NH Black": PALETTE["red"],\n                   "Hispanic": PALETTE["green"], "NH Asian": PALETTE["orange"]})\n\nplt.rcParams.update({"figure.figsize": (8.6, 4.8), "figure.dpi": 110,\n                     "axes.spines.top": False, "axes.spines.right": False,\n                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,\n                     "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,\n                     "axes.titlepad": 10, "font.family": "DejaVu Sans"})\n\ndef style(ax, title, xlabel, ylabel):\n    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)\n    return ax\n')

# The same Okabe-Ito palette and house style shared across every weekly notebook.
PALETTE = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
           "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "grey": "#999999",
           "ink": "#1a1a2e"}
RACE_ORDER = ["NH White", "NH Asian", "Hispanic", "NH Multiracial", "NH NHOPI", "NH AIAN", "NH Black"]
RACE_COLOR = {r: PALETTE["grey"] for r in RACE_ORDER}
RACE_COLOR.update({"NH White": PALETTE["blue"], "NH Black": PALETTE["red"],
                   "Hispanic": PALETTE["green"], "NH Asian": PALETTE["orange"]})

plt.rcParams.update({"figure.figsize": (8.6, 4.8), "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
                     "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
                     "axes.titlepad": 10, "font.family": "DejaVu Sans"})

def 

Next, the full-data year loader `load_years`, also from the Week 2 generator's SETUP. It scans the complete per-year cleaned files lazily and pools them, tagging each row with its `birth_year`. Weeks 2 and later use it for the full-population computations.

In [56]:
print('# The full-data year loader (Weeks 2+). Reads the complete per-year cleaned files lazily and pools\n# them, tagging each row with its birth_year. Do NOT call this on all years in this reference: it\n# materializes the ~24.9M-row, ~3GB full frame.\nINTERIM = Path("..") / "data" / "interim"\nYEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]\n\ndef load_years(years):\n    frames = [pl.scan_parquet(INTERIM / f"natality{y}_clean.parquet").with_columns(pl.lit(y).alias("birth_year"))\n              for y in years]\n    return pl.concat([f.collect() for f in frames], how="diagonal_relaxed")\n')

# The full-data year loader (Weeks 2+). Reads the complete per-year cleaned files lazily and pools
# them, tagging each row with its birth_year. Do NOT call this on all years in this reference: it
# materializes the ~24.9M-row, ~3GB full frame.
INTERIM = Path("..") / "data" / "interim"
YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

def load_years(years):
    frames = [pl.scan_parquet(INTERIM / f"natality{y}_clean.parquet").with_columns(pl.lit(y).alias("birth_year"))
              for y in years]
    return pl.concat([f.collect() for f in frames], how="diagonal_relaxed")



Finally, the two pandas EDA workhorses from the Week 1 generator's SETUP. `rate_by` is the grouping workhorse of the whole Week 1 exploration: preterm rate (%) and count for any grouping column. `risk_ratio_ci` computes the risk ratio of a binary flag (flag=1 vs flag=0) with a 95% log-normal confidence interval.

The CI derivation is worth spelling out, because it is the standard log-risk-ratio approach. With `a` preterm births among the `n1` flagged mothers and `c` among the `n0` unflagged, the two risks are `r1 = a/n1` and `r0 = c/n0`, and the risk ratio is `RR = r1/r0`. The sampling distribution of `RR` is skewed, so the interval is built on the log scale where it is approximately normal. The standard error of `log(RR)` is

`SE(log RR) = sqrt(1/a - 1/n1 + 1/c - 1/n0)`,

and the 95% interval is `RR * exp(-1.96 * SE)` to `RR * exp(+1.96 * SE)`, which is exactly what the function returns (alongside the two raw rates as percentages).

In [57]:
print('def rate_by(frame, col, dropna_target=True):\n    """Preterm rate (%) and count for any grouping column. The workhorse of the whole EDA."""\n    d = frame.dropna(subset=["preterm"]) if dropna_target else frame\n    g = d.groupby(col)["preterm"].agg(rate="mean", n="size")\n    g["rate"] = g["rate"] * 100\n    return g\n\ndef risk_ratio_ci(frame, flag_col, target="preterm"):\n    """Risk ratio (flag=1 vs flag=0) with a 95% log-normal CI. Returns (rr, lo, hi, r1, r0)."""\n    d = frame.dropna(subset=[target, flag_col])\n    g = d.groupby(flag_col)[target].agg(["mean", "sum", "size"])\n    r1, r0 = g.loc[1.0, "mean"], g.loc[0.0, "mean"]\n    a, n1 = g.loc[1.0, "sum"], g.loc[1.0, "size"]\n    c, n0 = g.loc[0.0, "sum"], g.loc[0.0, "size"]\n    rr = r1 / r0\n    se = np.sqrt(1/a - 1/n1 + 1/c - 1/n0)\n    return rr, rr*np.exp(-1.96*se), rr*np.exp(1.96*se), r1*100, r0*100\n')

def rate_by(frame, col, dropna_target=True):
    """Preterm rate (%) and count for any grouping column. The workhorse of the whole EDA."""
    d = frame.dropna(subset=["preterm"]) if dropna_target else frame
    g = d.groupby(col)["preterm"].agg(rate="mean", n="size")
    g["rate"] = g["rate"] * 100
    return g

def risk_ratio_ci(frame, flag_col, target="preterm"):
    """Risk ratio (flag=1 vs flag=0) with a 95% log-normal CI. Returns (rr, lo, hi, r1, r0)."""
    d = frame.dropna(subset=[target, flag_col])
    g = d.groupby(flag_col)[target].agg(["mean", "sum", "size"])
    r1, r0 = g.loc[1.0, "mean"], g.loc[0.0, "mean"]
    a, n1 = g.loc[1.0, "sum"], g.loc[1.0, "size"]
    c, n0 = g.loc[0.0, "sum"], g.loc[0.0, "size"]
    rr = r1 / r0
    se = np.sqrt(1/a - 1/n1 + 1/c - 1/n0)
    return rr, rr*np.exp(-1.96*se), rr*np.exp(1.96*se), r1*100, r0*100



<a id="map"></a>
## How it all fits, and the test map

Put together, the call graph runs from raw certificate to scored prediction to fairness audit:

1. **`ingest/schema.py`** defines the leakage classification (which raw columns are legitimate, which are barred, and why) and the per-variable sentinel codes. The ingest layer uses it to write the de-sentinelled per-year cleaned Parquet files.
2. **`features/build.py`** (`build_modeling_frame`) consumes a cleaned frame and produces the leakage-safe feature matrix `X`, target `y`, and component map, using `schema` for the classification and calling the **`features/leakage.py`** guard (`assert_no_component_is_excluded`) so no barred field can be laundered into a composite.
3. **`features/splits.py`** (`assign_splits`, `stratified_subsample`) partitions the rows deterministically and stratified, and draws the tractable working sample.
4. **`features/full_dataset.py`** composes the above: `outcome_audit` on the full census, and `assemble_modeling` (subsample -> `build_modeling_frame` -> `assign_splits`) into the modeling bundle.
5. **`models/baselines.py`** and **`models/boosting.py`** consume `X`, `y`, and the split masks to fit calibrated models and return held-out probabilities.
6. **`models/deep.py`** benchmarks TabNet and FT-Transformer on the same data.
7. **`models/ensemble.py`** stacks the base-model probabilities into a logistic meta-learner.
8. **`models/evaluate.py`** scores those probabilities with the calibration-first metrics and the tiered stratified report.
9. **`eval/robustness.py`** tests temporal stability via pre-COVID/post-COVID splits.
10. **`explain/shap_analysis.py`** turns the fitted tree model into global and race-stratified SHAP importance rankings.
11. **`fairness/decomposition.py`** g-computes how much of the Black-White gap is associated with measured covariates versus left unexplained.
12. **`fairness/mitigation.py`** runs ExponentiatedGradient and ThresholdOptimizer to price the fairness/accuracy tradeoff.

Every one of these modules is guarded by a test file, which is what makes the weekly notebooks safe to build on:

| Module | Test file |
| --- | --- |
| `ingest/schema.py` | `tests/test_ingest_schema.py` |
| `features/build.py` | `tests/test_features_build.py` |
| `features/leakage.py` | `tests/test_features_leakage.py` |
| `features/splits.py` | `tests/test_features_splits.py` |
| `features/full_dataset.py` | `tests/test_features_full_dataset.py` |
| `models/evaluate.py` | `tests/test_models_evaluate.py` |
| `models/baselines.py` | `tests/test_models_baselines.py` |
| `models/boosting.py` | `tests/test_models_boosting.py` |
| `models/deep.py` | `tests/test_models_deep.py` |
| `models/ensemble.py` | `tests/test_models_ensemble.py` |
| `eval/robustness.py` | `tests/test_eval_robustness.py` |
| `explain/shap_analysis.py` | `tests/test_explain_shap.py` |
| `fairness/decomposition.py` | `tests/test_fairness_mitigation.py` |
| `fairness/mitigation.py` | `tests/test_fairness_mitigation.py` |

That is the whole point of importing rather than re-implementing: the code you have just read is the code the tests cover, and the code the weekly notebooks run. Nothing in this reference can drift from either.

---

Author: [Parshv Patel](https://www.linkedin.com/in/parshv-patel-65a90326b/)  

--- 

Edit Logs:
